In [143]:
import json
import re
import requests

import pandas as pd

from bs4 import BeautifulSoup

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

# Extracción de datos

In [144]:
def extraer_informacion(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return []
    
    estate_json_raw = estate_component.get(":estate")
    estate_json_raw = estate_json_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_json_raw)

    data = {
        'dormitorios': estate_data.get('rooms'),
        'superficie_m2': estate_data.get('numeric_surface'),
        'baños': estate_data.get('bathrooms'),
        'url': estate_data.get('detail_url'),
        'features': estate_data.get('features'),
        'descripcion': estate_data.get('description'),
        'precio': estate_data.get('costs'),
        'latitud': estate_data.get('latitude'),
        'longitud': estate_data.get('longitude'),
        'media': estate_data.get('media'),
        'points_of_interest': estate_data.get('points_of_interest'),
        'energy_data': estate_data.get('energy_data'),
    }
    
    return data

In [145]:
def obtener_urls(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    print(f'Buscando pisos en la página {url} ...')
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    data = []

    for estate in estates_data:
        url_piso = estate.get("detail_url")

        if url_piso in df['url'].to_list():
            print('Ya lo tengo:', url_piso)
            return data
        data.append(extraer_informacion(url_piso))

    return data

In [146]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited))
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/guadalajara/yebes--valdeluz/647894.html


# Limpieza de datos

In [147]:
df

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data
0,2 dorm.,89.0,2 baños,https://www.tecnocasa.es/venta/piso/guadalajar...,"{'id': 647894, 'floor': 1, 'floors': None, 'bo...",<p>Agencia inmobiliaria de Yebes zona VALDELUZ...,"{'price': '216.000 €', 'box_price': None, 'mor...",40.591102,-3.110843,"{'floor_plans': None, 'has_realistico': True, ...",{'public_transport': [{'name': 'Guadalajara/Ye...,"{'class': 'e', 'class_emissions': 'f', 'certif..."
1,NaN,90.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 649701, 'floor': 2, 'floors': None, 'bo...","<p data-start=""90"" data-end=""251"">TECNOCASA GU...","{'price': '569.000 €', 'box_price': None, 'mor...",40.433602,-3.672523,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Diego de León'...,"{'class': 'e', 'class_emissions': 'e', 'certif..."
2,2 dorm.,65.0,2 baños,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 643217, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de Madrid - Chamberí -...,"{'price': '299.000 €', 'box_price': None, 'mor...",40.441002,-3.696503,"{'floor_plans': [{'id': 9485692, 'width': 1024...","{'public_transport': [{'name': 'Alonso Cano', ...","{'class': 'e', 'class_emissions': 'e', 'certif..."
3,3 dorm.,72.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mor...,"{'id': 645163, 'floor': 2, 'floors': None, 'bo...","<p data-start=""189"" data-end=""411"">La agencia ...","{'price': '220.000 €', 'box_price': None, 'mor...",40.677702,-3.966313,"{'floor_plans': [{'id': 9539314, 'width': 1154...",{'public_transport': [{'name': 'Estación de Au...,"{'class': 'e', 'class_emissions': 'e', 'certif..."
4,2 dorm.,76.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 646694, 'floor': '6 (planta semisótano)...",<p>Agencia inmobiliaria Tecnocasa Huerta Casta...,"{'price': '180.000 €', 'box_price': None, 'mor...",40.406300,-3.737990,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Alto de Extrem...,"{'class': 'g', 'class_emissions': 'g', 'certif..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1201,3 dorm.,100.0,2 baños,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 615767, 'floor': '', 'floors': None, 'b...","<p>Tecnocasa San Fermin, Estudio Inmobiliario ...","{'price': '300.000 €', 'box_price': None, 'mor...",40.365902,-3.698932,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'San Fermín-Orc...,"{'class': 'd', 'class_emissions': 'd', 'certif..."
1202,1 dorm.,72.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mec...,"{'id': 599314, 'floor': 'Baja', 'floors': None...",<p>AGENCIA INMOBILIARIA DE TECNOCASA EN MECO V...,"{'price': '129.500 €', 'box_price': None, 'mor...",40.549202,-3.324873,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Meco', 'class'...","{'class': 'c', 'class_emissions': 'b', 'certif..."
1203,NaN,54.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 600424, 'floor': '', 'floors': None, 'b...",<p>Descubre este acogedor piso en el barrio de...,"{'price': '225.000 €', 'box_price': None, 'mor...",40.428102,-3.638793,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Ascao', 'class...","{'class': 'e', 'class_emissions': 'e', 'certif..."
1204,NaN,66.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 615482, 'floor': 2, 'floors': None, 'bo...",<p>Descubre esta acogedora vivienda en el barr...,"{'price': '270.000 €', 'box_price': None, 'mor...",40.429402,-3.638183,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Ascao', 'class...","{'class': 'e', 'class_emissions': 'e', 'certif..."
